# Smart Lost & Found System — Experiment Notebook

This notebook reproduces the experiments in **Section 7 (Experimental Setup and Results)** of
*"Smart Lost & Found System for College Campus"*.

Since no real campus photographs are available, the notebook **generates a synthetic but
structurally realistic dataset**: each physical object gets a category, a colour, a distinguishing
texture pattern, a campus location, and a timestamp, and multiple photos of the same object are
rendered with different lighting/rotation/crop to imitate different phone cameras. A pretrained
CNN then extracts real visual embeddings from these images, and the fusion-matching pipeline from
the paper (visual + category + colour + location + time similarity) is run end-to-end on this data.

**Run all cells top to bottom** (Runtime → Run all). On a Colab CPU runtime this takes roughly
3–5 minutes; on a GPU runtime under a minute.

Everything below is computed live — the numbers your run produces are what should replace the
illustrative Table 5 / Table 6 / Table 7 values in the paper.

## 0. Setup

In [ ]:
!pip -q install torch torchvision --upgrade 2>/dev/null
import random, math, itertools
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image, ImageDraw, ImageFilter
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 1. Synthetic Campus Lost & Found Dataset

Category counts follow the composition used in the paper's Table 5: 200 objects / 600 images / 400
evaluation pairs. For every object:

- one image becomes the **lost report** photo,
- two further images (different augmentation = different photographer/lighting/angle) become
  **found report** query photos of the *same* physical object.

So each object contributes exactly one lost record and two found queries — 2 evaluation pairs per
object, matching the paper's "matching pairs" column exactly.

In [ ]:
CATEGORY_SPEC = {
    'Bag':          {'n_objects': 40, 'shape': 'rect_soft',  'palette': ['#1a1a1a', '#3b3f8f', '#7a1f1f', '#555555', '#4a3222']},
    'Bottle':       {'n_objects': 35, 'shape': 'ellipse',    'palette': ['#2f6fb0', '#e05c8a', '#3f9e5b', '#cccccc', '#e6b422']},
    'Phone':        {'n_objects': 25, 'shape': 'rounded',    'palette': ['#0a0a0a', '#eeeeee', '#3b5fa0', '#8a1f2b']},
    'Wallet':       {'n_objects': 20, 'shape': 'rect_hard',  'palette': ['#5b3a29', '#1a1a1a', '#a67c52']},
    'ID Card':      {'n_objects': 30, 'shape': 'card',       'palette': ['#2f6fb0', '#a01f1f', '#2f8f4f', '#d4a72c']},
    'Spectacles':   {'n_objects': 20, 'shape': 'glasses',    'palette': ['#1a1a1a', '#5b3a29', '#7a5230']},
    'Book':         {'n_objects': 30, 'shape': 'rect_hard',  'palette': ['#a01f1f', '#2f6fb0', '#2f8f4f', '#d4a72c', '#1a1a1a']},
}
IMG_SIZE = 160
ZONES = ['A', 'B', 'C', 'D', 'E']
BLOCKS = ['1', '2', '3']
SEMESTER_DAYS = 120

def hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def jitter_color(rgb, amount=18):
    return tuple(int(min(255, max(0, c + random.randint(-amount, amount)))) for c in rgb)

def draw_object(shape, rgb, texture_seed):
    img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (245, 245, 245))
    d = ImageDraw.Draw(img)
    rng = random.Random(texture_seed)
    cx, cy = IMG_SIZE // 2, IMG_SIZE // 2
    if shape == 'rect_soft':
        d.rounded_rectangle([30, 40, 130, 130], radius=18, fill=rgb, outline=(0, 0, 0))
    elif shape == 'ellipse':
        d.ellipse([50, 20, 110, 140], fill=rgb, outline=(0, 0, 0))
        d.rectangle([65, 10, 95, 30], fill=rgb, outline=(0, 0, 0))
    elif shape == 'rounded':
        d.rounded_rectangle([45, 15, 115, 145], radius=14, fill=rgb, outline=(0, 0, 0))
    elif shape == 'rect_hard':
        d.rectangle([30, 45, 130, 115], fill=rgb, outline=(0, 0, 0))
    elif shape == 'card':
        d.rounded_rectangle([25, 55, 135, 105], radius=6, fill=rgb, outline=(0, 0, 0))
    elif shape == 'glasses':
        d.ellipse([25, 60, 70, 100], outline=rgb, width=6)
        d.ellipse([90, 60, 135, 100], outline=rgb, width=6)
        d.line([70, 78, 90, 78], fill=rgb, width=6)
    # per-object distinguishing texture: a few deterministic speckles/lines
    for _ in range(6):
        x0 = rng.randint(20, IMG_SIZE - 20); y0 = rng.randint(20, IMG_SIZE - 20)
        d.ellipse([x0, y0, x0 + 3, y0 + 3], fill=(0, 0, 0))
    return img

def augment(img, rng):
    im = img.rotate(rng.uniform(-12, 12), fillcolor=(245, 245, 245), expand=False)
    enhancer_factor = rng.uniform(0.75, 1.25)
    arr = np.asarray(im).astype(np.float32) * enhancer_factor
    im = Image.fromarray(np.clip(arr, 0, 255).astype('uint8'))
    if rng.random() < 0.5:
        im = im.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0, 1.0)))
    # random crop-and-resize to simulate different framing
    pad = rng.randint(0, 12)
    im = im.crop((pad, pad, IMG_SIZE - pad, IMG_SIZE - pad)).resize((IMG_SIZE, IMG_SIZE))
    return im

records = []   # one row per PHOTO (lost report photo or found query photo)
obj_id = 0
for category, spec in CATEGORY_SPEC.items():
    for _ in range(spec['n_objects']):
        base_color = hex_to_rgb(random.choice(spec['palette']))
        texture_seed = obj_id * 97 + 13
        base_img = draw_object(spec['shape'], jitter_color(base_color, 6), texture_seed)

        lost_zone = random.choice(ZONES); lost_block = random.choice(BLOCKS)
        lost_time = random.uniform(0, SEMESTER_DAYS - 10)
        rng_lost = random.Random(texture_seed + 1)
        lost_photo = augment(base_img, rng_lost)
        records.append(dict(obj_id=obj_id, category=category, role='lost', img=lost_photo,
                             color_rgb=base_color, zone=lost_zone, block=lost_block, room=str(random.randint(1, 20)),
                             time=lost_time))

        for k in range(2):  # two found-report queries of the same physical object
            r = random.random()
            if r < 0.6:
                f_zone, f_block, f_room = lost_zone, lost_block, records[-1]['room']
            elif r < 0.85:
                f_zone, f_block, f_room = lost_zone, lost_block, str(random.randint(1, 20))
            else:
                f_zone, f_block, f_room = random.choice(ZONES), random.choice(BLOCKS), str(random.randint(1, 20))
            found_time = lost_time + random.uniform(0.2, 10)
            rng_found = random.Random(texture_seed + 10 + k)
            found_photo = augment(base_img, rng_found)
            records.append(dict(obj_id=obj_id, category=category, role='found', img=found_photo,
                                 color_rgb=base_color, zone=f_zone, block=f_block, room=f_room,
                                 time=found_time))
        obj_id += 1

df = pd.DataFrame(records)
print('Total objects:', obj_id)
print('Total images:', len(df), '(lost:', (df.role=='lost').sum(), ', found:', (df.role=='found').sum(), ')')
df.groupby('category').agg(objects=('obj_id', 'nunique'), images=('obj_id', 'count')).assign(
    matching_pairs=lambda t: t['objects'] * 2)

**Sanity check — sample rendered photos:**

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
sample = df.sample(12, random_state=1).reset_index(drop=True)
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    ax.imshow(row['img']); ax.set_title(f"{row['category']}\n{row['role']}", fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Visual Feature Extraction

A ResNet-18 pretrained on ImageNet is used as the backbone (Section 5.2 of the paper). The final
pooling-layer activations are taken as the descriptor and L2-normalized, so cosine similarity
lies in [0, 1] for these non-negative post-ReLU features.

In [ ]:
backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
backbone.fc = nn.Identity()
backbone.eval().to(device)

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def embed_images(pil_images, batch_size=64):
    vecs = []
    for i in range(0, len(pil_images), batch_size):
        batch = torch.stack([preprocess(im) for im in pil_images[i:i+batch_size]]).to(device)
        v = backbone(batch)
        v = torch.nn.functional.normalize(v, dim=1)
        vecs.append(v.cpu().numpy())
    return np.concatenate(vecs, axis=0)

embeddings = embed_images(list(df['img']))
df['emb_idx'] = np.arange(len(df))
print('Embedding matrix:', embeddings.shape)

## 3. Attribute Similarity Components (Section 5.3)

- **Category** — exact match required; candidate retrieval is restricted to the same category, as
  the paper specifies ("acts as a filter against cross-category false matches").
- **Colour** — mean colour is compared in Lab-like Euclidean space between the query's true colour
  and each candidate's true colour (a stand-in for the dominant-colour extraction described in the
  paper).
- **Location** — same room = 1.0, same block = 0.6, same zone = 0.3, different zone = 0.05.
- **Time** — exponential decay `exp(-|Δt|/τ)`, zero if the found event precedes the lost event.

The paper's sixth component (free-text description similarity) has no counterpart here, since the
synthetic reports carry no free text; its weight is redistributed proportionally across the other
five, matching the paper's rule for reports that are missing a modality.

In [ ]:
def rgb_dist_similarity(rgb_a, rgb_b, scale=140.0):
    d = math.sqrt(sum((a - b) ** 2 for a, b in zip(rgb_a, rgb_b)))
    return max(0.0, 1.0 - d / scale)

LOCATION_SCORES = {'room': 1.0, 'block': 0.6, 'zone': 0.3, 'other': 0.05}
def location_similarity(a, b):
    if a['zone'] != b['zone']:
        return LOCATION_SCORES['other']
    if a['block'] != b['block']:
        return LOCATION_SCORES['zone']
    if a['room'] != b['room']:
        return LOCATION_SCORES['block']
    return LOCATION_SCORES['room']

TAU_DAYS = 5.0
def time_similarity(t_found, t_lost, eps=0.05):
    dt = t_found - t_lost
    if dt < -eps:
        return 0.0
    return math.exp(-abs(dt) / TAU_DAYS)

# weights from Table 3, renormalized after dropping the text component (0.10)
W_RAW = dict(image=0.40, category=0.15, colour=0.10, location=0.15, time=0.10)
_scale = 1.0 / (1.0 - 0.10)
W = {k: v * _scale for k, v in W_RAW.items()}
print('Renormalized fusion weights:', {k: round(v, 3) for k, v in W.items()})

# attribute-only weights (renormalize the four non-visual components to sum to 1)
_attr_sum = W['category'] + W['colour'] + W['location'] + W['time']
W_ATTR = {k: W[k] / _attr_sum for k in ['category', 'colour', 'location', 'time']}
print('Attribute-only weights:', {k: round(v, 3) for k, v in W_ATTR.items()})

## 4. Matching Pipeline and Score Fusion (Section 5.4)

For every found-report query, candidates are the **lost reports of the same category** (the
category filter from Section 5.3). Three configurations are scored per candidate pair:

`Score = w_image·S_image + w_category·S_category + w_colour·S_colour + w_location·S_location + w_time·S_time`

- **visual-only** — `S_image` alone.
- **attribute-only** — the four non-visual components with `W_ATTR`.
- **fused** — the full weighted combination with `W`.

In [ ]:
lost_df = df[df.role == 'lost'].reset_index(drop=True)
found_df = df[df.role == 'found'].reset_index(drop=True)

results = []  # one row per found query, per configuration
for _, q in found_df.iterrows():
    cands = lost_df[lost_df.category == q['category']]
    q_emb = embeddings[q['emb_idx']]
    s_image = embeddings[cands['emb_idx'].values] @ q_emb  # cosine sim (already normalized)
    s_category = np.ones(len(cands))  # candidates are already category-filtered
    s_colour = np.array([rgb_dist_similarity(q['color_rgb'], c) for c in cands['color_rgb']])
    s_location = np.array([location_similarity(q, c) for _, c in cands.iterrows()])
    s_time = np.array([time_similarity(q['time'], t) for t in cands['time']])

    score_visual = s_image
    score_attr = (W_ATTR['category'] * s_category + W_ATTR['colour'] * s_colour +
                  W_ATTR['location'] * s_location + W_ATTR['time'] * s_time)
    score_fused = (W['image'] * s_image + W['category'] * s_category + W['colour'] * s_colour +
                   W['location'] * s_location + W['time'] * s_time)

    true_obj = q['obj_id']
    for config_name, scores in [('visual', score_visual), ('attribute', score_attr), ('fused', score_fused)]:
        order = np.argsort(-scores)
        ranked_obj_ids = cands['obj_id'].values[order]
        ranked_scores = scores[order]
        rank_of_true = int(np.where(ranked_obj_ids == true_obj)[0][0]) + 1  # 1-indexed
        results.append(dict(query_idx=q['emb_idx'], config=config_name, true_obj=true_obj,
                             top1_obj=ranked_obj_ids[0], top1_score=ranked_scores[0],
                             rank_of_true=rank_of_true, n_candidates=len(cands)))

res_df = pd.DataFrame(results)
res_df.head()

## 5. Evaluation Metrics (Section 7.2)

For each configuration, a decision threshold on the top-1 score is swept to find the value that
maximizes F1: a query is "declared a match" if its top-1 score clears the threshold; it counts as
correct only if the top-1 candidate is truly the same object.

In [ ]:
def best_threshold_metrics(sub_df):
    thresholds = np.linspace(0, 1, 201)
    best = {'f1': -1}
    correct = (sub_df['top1_obj'] == sub_df['true_obj']).values
    scores = sub_df['top1_score'].values
    for t in thresholds:
        declared = scores >= t
        tp = np.sum(declared & correct)
        fp = np.sum(declared & ~correct)
        fn = np.sum(~declared)  # a query with a true match that wasn't declared (correctly or not)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        if f1 > best['f1']:
            best = dict(threshold=t, precision=precision, recall=recall, f1=f1)
    return best

def topk_accuracy(sub_df, ks=(1, 3, 5, 10)):
    return {k: float((sub_df['rank_of_true'] <= k).mean()) for k in ks}

table6_rows = []
for config_name, label in [('visual', 'Visual features only'), ('attribute', 'Attributes only'), ('fused', 'Proposed fused model')]:
    sub = res_df[res_df.config == config_name]
    m = best_threshold_metrics(sub)
    top5 = topk_accuracy(sub, ks=(5,))[5]
    table6_rows.append(dict(Configuration=label, Precision=round(m['precision'], 2),
                             Recall=round(m['recall'], 2), **{'F1-score': round(m['f1'], 2)},
                             **{'Top-5 accuracy': round(top5, 2)}))

table6 = pd.DataFrame(table6_rows)
print('Table 6 — Matching performance by configuration')
table6

In [ ]:
fused_sub = res_df[res_df.config == 'fused']
topk = topk_accuracy(fused_sub, ks=(1, 3, 5, 10))
table7 = pd.DataFrame({'k': list(topk.keys()), 'Accuracy': [round(v, 2) for v in topk.values()]})
print('Table 7 — Top-k retrieval accuracy of the fused model')
table7

In [ ]:
# Table 5 recomputed directly from the generated dataset (should match Section 7.1 exactly)
table5 = df[df.role == 'lost'].groupby('category').size().rename('objects').to_frame()
table5['images'] = df.groupby('category').size()
table5['matching_pairs'] = table5['objects'] * 2
table5.loc['Total'] = table5.sum()
print('Table 5 — Composition of the evaluation dataset')
table5

**Per-category breakdown of the fused model's top-1 accuracy** (Section 7.4 — uniform, mass-produced categories are expected to score lowest):

In [ ]:
fused_with_cat = fused_sub.merge(df[df.role=='found'][['emb_idx','category']], left_on='query_idx', right_on='emb_idx')
per_cat = fused_with_cat.assign(correct=lambda t: t['rank_of_true'] == 1).groupby('category')['correct'].mean().sort_values()
per_cat.to_frame('Top-1 accuracy').round(2)

**Visual comparison of the three configurations:**

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
table6.set_index('Configuration')[['Precision', 'Recall', 'F1-score', 'Top-5 accuracy']].plot(kind='bar', ax=ax)
ax.set_ylim(0, 1); ax.set_ylabel('Score'); ax.set_title('Matching performance by configuration')
plt.xticks(rotation=15, ha='right'); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(list(topk.keys()), list(topk.values()), marker='o')
ax.set_xlabel('k'); ax.set_ylabel('Top-k accuracy'); ax.set_ylim(0, 1)
ax.set_title('Top-k retrieval accuracy — fused model'); ax.set_xticks(list(topk.keys()))
plt.tight_layout(); plt.show()

## 6. Notes on Using These Numbers in the Paper

- Every table above is computed live from this run; re-running the notebook (especially after
  changing `SEED`) will shift the exact figures slightly, so report the numbers from **one fixed
  run** and mention the seed.
- This is a synthetic proxy dataset (rendered shapes, not real campus photographs), built to match
  the paper's Table 5 composition exactly. Replace the dataset-generation cell in Section 1 with a
  loader over real photographs (grouped into `lost/` and `found/` folders per object) to get results
  directly transferable to the paper without the "synthetic" caveat.
- The free-text description similarity component (`S_text` in Section 5.3) is not modelled here; if
  your real reports include descriptions, add a sentence-embedding similarity column and fold its
  weight back into `W` instead of redistributing it.
- To match the paper's Table 6 wording, round each metric to two decimal places as done above.
